# Task 3: Historical Reward Engine for First-Time VW Training

This notebook builds reward labels from historical `campaign_sent` events. It does not require production decision logs.

Key contracts:
- all campaign sends stay in the label table, including zero-outcome sends
- `net_reward = decayed_reward - channel_cost`
- `vw_cost` is a deterministic bounded cost in `[0, 1]`, where lower is better
- outputs are written to `rewards/reward_labels_<USE_CASE_ID>_LOG_SCALED.parquet` and `rewards/event_audit_<USE_CASE_ID>.parquet`


In [ ]:
import io
import json
import os
import warnings
from datetime import timedelta

import boto3
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

print('=' * 80)
print('TASK 3: HISTORICAL REWARD ENGINE - SETUP AND DATA LOADING')
print('FIRST-TIME MODE: using historical campaign_sent events, not decision logs')
print('=' * 80)


def get_use_case_id(default: str = 'pl-aip-uplift') -> str:
    value = os.getenv('USE_CASE_ID', default).strip()
    if not value:
        raise ValueError('USE_CASE_ID must be non-empty.')
    return value


def to_utc_naive(series) -> pd.Series:
    return pd.to_datetime(series, errors='coerce', utc=True).dt.tz_convert(None)


def read_json_s3(bucket: str, key: str) -> dict:
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return json.loads(obj['Body'].read().decode('utf-8'))


def read_parquet_s3(bucket: str, key: str) -> pd.DataFrame:
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(io.BytesIO(obj['Body'].read()))


def normalize_channel(value) -> str:
    channel_map = {
        'CH001': 'whatsapp', 'WHATSAPP': 'whatsapp', 'WA': 'whatsapp',
        'CH002': 'sms', 'SMS': 'sms',
        'CH003': 'rcs', 'RCS': 'rcs', 'GRCS': 'rcs',
        'CH004': 'push', 'PUSH': 'push', 'BPN': 'push', 'APN': 'push',
        'CH005': 'email', 'EMAIL': 'email',
    }
    raw = '' if pd.isna(value) else str(value).strip()
    return channel_map.get(raw.upper(), raw.lower())


USE_CASE_ID = get_use_case_id('pl-aip-uplift')
S3_CONFIG_BUCKET = os.getenv('S3_CONFIG_BUCKET', 'aks-nvtabular-data')
S3_DATA_BUCKET = os.getenv('S3_DATA_BUCKET', 'pratham-nvtabular-poc-data')
REWARD_CONFIG_KEY = os.getenv('REWARD_CONFIG_KEY', 'config/reward_config.json')
FUNNEL_CONFIG_KEY = os.getenv('FUNNEL_CONFIG_KEY', 'config/funnel_config.json')
S3_DATA_KEY = os.getenv('S3_DATA_KEY', 'logical_synthetic_data/processed/raw_splits/full_df.parquet')
s3_client = boto3.client('s3')

print('[1/3] Loading reward and funnel configs...')
REWARD_CONFIG = read_json_s3(S3_CONFIG_BUCKET, REWARD_CONFIG_KEY)
FUNNEL_CONFIG = read_json_s3(S3_CONFIG_BUCKET, FUNNEL_CONFIG_KEY)
reward_events = REWARD_CONFIG.get('reward_events', {})
if not reward_events:
    raise ValueError('reward_config.json must contain non-empty reward_events.')
print(f'  Reward events: {len(reward_events):,}')
print(f'  Funnel stages: {len(FUNNEL_CONFIG.get("funnel_stages", [])):,}')

print('[2/3] Loading historical event data...')
df_events = read_parquet_s3(S3_DATA_BUCKET, S3_DATA_KEY)
if 'master_user_id' not in df_events.columns and 'customer_id' in df_events.columns:
    df_events['master_user_id'] = df_events['customer_id']
required_event_cols = ['event_name', 'timestamp', 'master_user_id']
missing_event_cols = [c for c in required_event_cols if c not in df_events.columns]
if missing_event_cols:
    raise ValueError(f'Historical event data missing required columns: {missing_event_cols}')

df_events = df_events.copy()
df_events['master_user_id'] = df_events['master_user_id'].astype(str)
df_events['timestamp'] = to_utc_naive(df_events['timestamp'])
if df_events['timestamp'].isna().any():
    raise ValueError('Historical event data contains invalid timestamps.')
print(f'  Raw event rows: {len(df_events):,}')
print(f'  Date range    : {df_events["timestamp"].min()} -> {df_events["timestamp"].max()}')

print('[3/3] Building campaign_sent anchor table...')
df_campaign_sent = df_events.loc[df_events['event_name'].astype(str) == 'campaign_sent'].copy()
if df_campaign_sent.empty:
    raise ValueError("No campaign_sent rows found. First-time CB training needs historical sends as logged actions.")
if 'campaign_id' not in df_campaign_sent.columns:
    raise ValueError('campaign_sent rows must contain campaign_id.')
if 'channel_id' not in df_campaign_sent.columns:
    if 'channel' not in df_campaign_sent.columns:
        raise ValueError('campaign_sent rows must contain channel_id or channel.')
    df_campaign_sent['channel_id'] = df_campaign_sent['channel']
df_campaign_sent['channel_id'] = df_campaign_sent['channel_id'].apply(normalize_channel)

passthrough_cols = [
    'campaign_id', 'master_user_id', 'timestamp', 'channel_id', 'customer_id',
    'action_id', 'recommended_action_id', 'channel', 'day', 'offer', 'time',
]
passthrough_cols = [c for c in passthrough_cols if c in df_campaign_sent.columns]
df_campaign_sent = df_campaign_sent[passthrough_cols].copy()
df_campaign_sent = df_campaign_sent.sort_values(['master_user_id', 'timestamp', 'campaign_id']).reset_index(drop=True)
print(f'  Campaign anchors: {len(df_campaign_sent):,}')
print(f'  Unique users    : {df_campaign_sent["master_user_id"].nunique():,}')
print(f'  Unique campaigns: {df_campaign_sent["campaign_id"].nunique():,}')
print('=' * 80)


In [ ]:
print('=' * 80)
print('TASK 3: EVENT ATTRIBUTION AUDIT')
print('=' * 80)

event_rewards = {}
event_funnel_pos = {}
event_stage = {}
for event_name, event_config in REWARD_CONFIG['reward_events'].items():
    if 'reward' in event_config:
        event_rewards[event_name] = event_config['reward']
    elif 'reward_column' in event_config:
        event_rewards[event_name] = 'event_value'
    else:
        event_rewards[event_name] = 0.0
    event_funnel_pos[event_name] = event_config.get('funnel_position')
    event_stage[event_name] = event_config.get('funnel_stage', 'unknown')

attribution_window_days = int(REWARD_CONFIG.get('attribution_window_days', 7))
avg_conversion_value = float(REWARD_CONFIG.get('avg_conversion_value', FUNNEL_CONFIG.get('avg_conversion_value', 129905)))

campaigns = df_campaign_sent.sort_values(['master_user_id', 'timestamp', 'campaign_id']).reset_index(drop=True).copy()
campaigns['next_campaign_timestamp'] = campaigns.groupby('master_user_id')['timestamp'].shift(-1)
user_event_groups = {
    user_id: grp.sort_values('timestamp')
    for user_id, grp in df_events.groupby('master_user_id', sort=False)
}

audit_rows = []
print(f'Attributing events for {len(campaigns):,} campaign sends...')
for idx, row in enumerate(campaigns.itertuples(index=False), start=1):
    row_dict = row._asdict()
    campaign_id = row_dict['campaign_id']
    user_id = row_dict['master_user_id']
    campaign_timestamp = row_dict['timestamp']
    channel_id = row_dict['channel_id']

    nominal_window_end = campaign_timestamp + timedelta(days=attribution_window_days)
    next_campaign_ts = row_dict.get('next_campaign_timestamp')
    window_end = min(nominal_window_end, next_campaign_ts) if pd.notna(next_campaign_ts) else nominal_window_end

    user_events = user_event_groups.get(user_id)
    if user_events is None:
        continue
    mask = (user_events['timestamp'] > campaign_timestamp) & (user_events['timestamp'] <= window_end)
    user_events_in_window = user_events.loc[mask]

    cumulative_reward = 0.0
    for event_row in user_events_in_window.itertuples(index=False):
        event_name = getattr(event_row, 'event_name')
        if event_name not in event_rewards:
            continue
        reward_value = event_rewards[event_name]
        if reward_value == 'event_value':
            raw_value = getattr(event_row, 'event_value', np.nan)
            actual_reward = float(raw_value) if pd.notna(raw_value) else avg_conversion_value
        else:
            actual_reward = float(reward_value)
        cumulative_reward += actual_reward
        event_timestamp = getattr(event_row, 'timestamp')
        audit_rows.append({
            'campaign_id': campaign_id,
            'master_user_id': user_id,
            'campaign_timestamp': campaign_timestamp,
            'channel_id': channel_id,
            'event_name': event_name,
            'event_timestamp': event_timestamp,
            'event_stage': event_stage.get(event_name, 'unknown'),
            'funnel_position': event_funnel_pos.get(event_name, 0) or 0,
            'event_reward': actual_reward,
            'is_reward_event': int(actual_reward != 0.0),
            'is_positive_reward_event': int(actual_reward > 0.0),
            'cumulative_reward': cumulative_reward,
            'time_since_campaign_hours': (event_timestamp - campaign_timestamp).total_seconds() / 3600.0,
            'time_since_campaign_days': (event_timestamp - campaign_timestamp).total_seconds() / 86400.0,
            'window_end': window_end,
        })
    if idx % 5000 == 0 or idx == len(campaigns):
        print(f'  Processed {idx:,} / {len(campaigns):,}')

audit_cols = [
    'campaign_id', 'master_user_id', 'campaign_timestamp', 'channel_id', 'event_name',
    'event_timestamp', 'event_stage', 'funnel_position', 'event_reward', 'is_reward_event',
    'is_positive_reward_event', 'cumulative_reward', 'time_since_campaign_hours',
    'time_since_campaign_days', 'window_end',
]
df_event_audit = pd.DataFrame(audit_rows, columns=audit_cols)
print(f'Attributed reward events: {len(df_event_audit):,}')
print('=' * 80)


In [ ]:
print('=' * 80)
print('TASK 3: CAMPAIGN-LEVEL REWARD AND VW COST')
print('=' * 80)

channel_costs = REWARD_CONFIG.get('channel_costs', {
    'whatsapp': 0.39,
    'sms': 0.12,
    'rcs': 0.25,
    'push': 0.03,
    'email': 0.05,
})
channel_costs = {str(k).lower(): float(v) for k, v in channel_costs.items()}
time_decay_rate = float(REWARD_CONFIG.get('time_decay_rate', 0.9))
avg_conversion_value = float(REWARD_CONFIG.get('avg_conversion_value', FUNNEL_CONFIG.get('avg_conversion_value', 129905)))

campaign_base = df_campaign_sent.rename(columns={'timestamp': 'campaign_timestamp'}).copy()
base_keys = ['campaign_id', 'master_user_id', 'campaign_timestamp', 'channel_id']

if df_event_audit.empty:
    df_output_a = campaign_base.copy()
    df_output_a['events_found_all'] = 0
    df_output_a['events_found_rewarding'] = 0
    df_output_a['events_found_positive'] = 0
    df_output_a['events_found'] = 0
    df_output_a['total_event_reward'] = 0.0
    df_output_a['max_funnel_position'] = 0
    df_output_a['attribution_delay_hours'] = 0.0
    df_output_a['attribution_delay_days'] = 0.0
    df_output_a['time_decay_factor'] = 1.0
    df_output_a['decayed_reward'] = 0.0
else:
    agg = df_event_audit.groupby(base_keys, as_index=False, dropna=False).agg(
        events_found_all=('event_name', 'count'),
        events_found_rewarding=('is_reward_event', 'sum'),
        events_found_positive=('is_positive_reward_event', 'sum'),
        total_event_reward=('event_reward', 'sum'),
        max_funnel_position=('funnel_position', 'max'),
        last_event_timestamp=('event_timestamp', 'max'),
    )
    delta_seconds = (agg['last_event_timestamp'] - agg['campaign_timestamp']).dt.total_seconds().clip(lower=0.0)
    agg['attribution_delay_hours'] = delta_seconds / 3600.0
    agg['attribution_delay_days'] = delta_seconds / 86400.0
    agg['time_decay_factor'] = np.power(time_decay_rate, agg['attribution_delay_days']).clip(0.0, 1.0)
    agg['decayed_reward'] = agg['total_event_reward'] * agg['time_decay_factor']
    agg['events_found'] = agg['events_found_rewarding']
    agg = agg.drop(columns=['last_event_timestamp'])
    df_output_a = campaign_base.merge(agg, on=base_keys, how='left')
    for col in ['events_found_all', 'events_found_rewarding', 'events_found_positive', 'events_found', 'max_funnel_position']:
        df_output_a[col] = df_output_a[col].fillna(0).astype(int)
    for col in ['total_event_reward', 'attribution_delay_hours', 'attribution_delay_days', 'decayed_reward']:
        df_output_a[col] = df_output_a[col].fillna(0.0)
    df_output_a['time_decay_factor'] = df_output_a['time_decay_factor'].fillna(1.0)

missing_cost_channels = sorted(set(df_output_a['channel_id'].astype(str).str.lower()) - set(channel_costs))
if missing_cost_channels:
    raise ValueError(f'Missing channel costs for channel_id values: {missing_cost_channels}')
df_output_a['channel_id'] = df_output_a['channel_id'].astype(str).str.lower()
df_output_a['channel_cost'] = df_output_a['channel_id'].map(channel_costs)
df_output_a['net_reward'] = df_output_a['decayed_reward'] - df_output_a['channel_cost']

reward_floor = float(REWARD_CONFIG.get('vw_cost_reward_floor', -max(channel_costs.values())))
reward_ceiling = min(float(REWARD_CONFIG.get('vw_cost_reward_ceiling', avg_conversion_value)), 5000.0)
if not np.isfinite(reward_floor) or not np.isfinite(reward_ceiling) or reward_ceiling <= reward_floor:
    raise ValueError(f'Invalid VW cost reward bounds: floor={reward_floor}, ceiling={reward_ceiling}')

bounded_reward = df_output_a['net_reward'].clip(lower=reward_floor, upper=reward_ceiling)
reward_offset = bounded_reward - reward_floor
reward_span = reward_ceiling - reward_floor
df_output_a['reward_floor'] = reward_floor
df_output_a['reward_ceiling'] = reward_ceiling
df_output_a['bounded_reward'] = bounded_reward
df_output_a['scaled_reward'] = np.log1p(reward_offset) / np.log1p(reward_span)
df_output_a['vw_cost'] = (1.0 - df_output_a['scaled_reward']).clip(0.0, 1.0)

if df_output_a['vw_cost'].isna().any() or not df_output_a['vw_cost'].between(0, 1, inclusive='both').all():
    raise ValueError('vw_cost must be finite and in [0, 1].')
formula_delta = (df_output_a['net_reward'] - (df_output_a['decayed_reward'] - df_output_a['channel_cost'])).abs().max()
if formula_delta > 1e-9:
    raise ValueError('net_reward formula validation failed.')

columns_to_drop = ['bounded_reward', 'scaled_reward']
df_output_a_clean = df_output_a.drop(columns=columns_to_drop, errors='ignore')

print(f'Campaign reward rows : {len(df_output_a_clean):,}')
print(f'Zero-event rows      : {(df_output_a_clean["events_found_all"] == 0).sum():,}')
print(f'VW cost range        : {df_output_a_clean["vw_cost"].min():.6f} - {df_output_a_clean["vw_cost"].max():.6f}')
print(f'Reward bounds        : floor={reward_floor:.4f}, ceiling={reward_ceiling:.4f}')
print(df_output_a_clean[['campaign_id', 'master_user_id', 'channel_id', 'net_reward', 'vw_cost']].head(5).to_string(index=False))
print('=' * 80)


In [ ]:
print('=' * 80)
print('TASK 3: UPLOAD HISTORICAL REWARD OUTPUTS TO S3')
print('=' * 80)

reward_key = f'rewards/reward_labels_{USE_CASE_ID}_LOG_SCALED.parquet'
audit_key = f'rewards/event_audit_{USE_CASE_ID}.parquet'
try:
    reward_buffer = io.BytesIO()
    df_output_a_clean.to_parquet(reward_buffer, index=False)
    reward_buffer.seek(0)
    s3_client.upload_fileobj(reward_buffer, S3_CONFIG_BUCKET, reward_key)

    audit_buffer = io.BytesIO()
    df_event_audit.to_parquet(audit_buffer, index=False)
    audit_buffer.seek(0)
    s3_client.upload_fileobj(audit_buffer, S3_CONFIG_BUCKET, audit_key)
except Exception as exc:
    print(f'ERROR uploading Task 3 outputs: {exc}')
    raise

print(f'Reward labels: s3://{S3_CONFIG_BUCKET}/{reward_key}')
print(f'Event audit  : s3://{S3_CONFIG_BUCKET}/{audit_key}')
print('TASK 3 COMPLETE')
print('=' * 80)
